# Ensemble/Voting Classification in Python with Scikit-Learn
ref：https://www.kaggle.com/c/titanic/submit

In [41]:
import pandas as pd
import numpy as np
import warnings

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.model_selection import train_test_split, KFold, cross_val_score

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, ExtraTreesClassifier

In [42]:
# training_dat = ???
# testing_data = ???
# def get_nulls(training, testing):
#     print("Training Data:")
#     print(pd.isnull(training).sum())
#     print("Testing Data:")
#     print(pd.isnull(testing).sum())

# get_nulls(training_data, testing_data)


import pandas as pd

# 1. 讀取訓練資料與測試資料
training_data = pd.read_csv("data/train.csv")
testing_data = pd.read_csv("data/test.csv")

# 2. 建立函數，檢查資料中的缺失值
def get_nulls(training, testing):

    # 顯示訓練資料各欄位的缺失值數量
    print("Training Data:")
    print(pd.isnull(training).sum())

    # 顯示測試資料各欄位的缺失值數量
    print("Testing Data:")
    print(pd.isnull(testing).sum())

# 3. 呼叫函數，檢查兩份資料
get_nulls(training_data, testing_data)



Training Data:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
Testing Data:
PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64


In [43]:
# # Drop the cabin column, as there are too many missing values
# # Drop the ticket numbers too, as there are too many categories
# # Drop names as they won't really help predict survivors
# # Taking the mean/average value would be impacted by the skew
# # so we should use the median value to impute missing values
# training_data["Age"].fillna(training_data["Age"].median(),inplace=True)
# get_nulls(training_data, testing_data)
# 刪除訓練資料中的三個欄位
training_data.drop(
    labels=['Cabin', 'Ticket', 'Name'],
    axis=1,
    inplace=True
)
# 刪除測試資料中的三個欄位
testing_data.drop(
    labels=['Cabin', 'Ticket', 'Name'],
    axis=1,
    inplace=True
)
# 1. 使用訓練資料 Age 的中位數填補缺失值
training_data["Age"] = training_data["Age"].fillna(
    training_data["Age"].median()
)
# 2. 使用測試資料 Age 的中位數填補缺失值
testing_data["Age"] = testing_data["Age"].fillna(
    testing_data["Age"].median()
)
# 3. 將訓練資料 Embarked 的缺失值填入 "S"
# S 代表 Southampton（南安普敦港）
training_data["Embarked"] = training_data["Embarked"].fillna("S")
# 4. 使用測試資料 Fare 的中位數填補缺失值
testing_data["Fare"] = testing_data["Fare"].fillna(
    testing_data["Fare"].median()
)
# 5. 再次檢查缺失值
get_nulls(training_data, testing_data)

Training Data:
PassengerId    0
Survived       0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       0
dtype: int64
Testing Data:
PassengerId    0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       0
dtype: int64


In [44]:
# Fit the encoder on the data (Feature: Sex)
# Transform and replace training data
# Fit the encoder on the data (Feature: Embarked)
# Any value we want to reshape needs be turned into array first
# Scaler takes arrays

In [45]:

# ==========================================
# 1. Label Encoding：將 Sex（性別）轉換成數值
# ==========================================

# 建立 LabelEncoder 編碼器
encoder_1 = LabelEncoder()

# 使用訓練資料中的 Sex 欄位建立編碼規則
encoder_1.fit(training_data["Sex"])

# 將訓練資料 Sex 轉換成數值
training_sex_encoded = encoder_1.transform(training_data["Sex"])

# 將編碼結果更新回訓練資料
training_data["Sex"] = training_sex_encoded

# 使用相同編碼器轉換測試資料
test_sex_encoded = encoder_1.transform(testing_data["Sex"])

# 將編碼結果更新回測試資料
testing_data["Sex"] = test_sex_encoded


# ==========================================
# 2. Label Encoding：將 Embarked（登船港口）轉換成數值
# ==========================================

# 建立第二個 LabelEncoder
encoder_2 = LabelEncoder()

# 根據訓練資料 Embarked 建立編碼規則
encoder_2.fit(training_data["Embarked"])

# 將訓練資料 Embarked 轉換成數值
training_embarked_encoded = encoder_2.transform(training_data["Embarked"])

# 更新訓練資料
training_data["Embarked"] = training_embarked_encoded

# 使用相同的編碼器轉換測試資料
testing_embarked_encoded = encoder_2.transform(testing_data["Embarked"])

# 更新測試資料
testing_data["Embarked"] = testing_embarked_encoded


# ==========================================
# 3. StandardScaler：標準化 Age（年齡）
# ==========================================

# 將 Age 轉成 NumPy 陣列，並調整成二維
ages_train = np.array(training_data["Age"]).reshape(-1, 1)

# 建立標準化工具
scaler = StandardScaler()

# 使用訓練資料的平均數與標準差進行標準化
training_data["Age"] = scaler.fit_transform(ages_train).ravel()


In [46]:
# Now to select our training/testing data
# Make the train/test data from validation

# ==========================================
# 1. 選擇模型訓練需要的特徵（Features）
# ==========================================

# 移除 PassengerId（乘客編號）及 Survived（是否生還）
# 其餘欄位作為模型輸入特徵 X
X_features = training_data.drop(
    labels=['PassengerId', 'Survived'],
    axis=1
)

# 將 Survived 作為預測目標 y
# 0 = 未生還，1 = 生還
y_labels = training_data['Survived']

# 顯示前 5 筆特徵資料
print(X_features.head(5))


# ==========================================
# 2. 將資料分割為訓練集與驗證集
# ==========================================

# 90% 用於訓練，10% 用於驗證模型
# random_state=12：固定隨機種子，確保結果可重現
X_train, X_val, y_train, y_val = train_test_split(
    X_features,
    y_labels,
    test_size=0.1,
    random_state=12
)


   Pclass  Sex       Age  SibSp  Parch     Fare  Embarked
0       3    1 -0.565736      1      0   7.2500         2
1       1    0  0.663861      1      0  71.2833         0
2       3    0 -0.258337      0      0   7.9250         2
3       1    0  0.433312      1      0  53.1000         2
4       3    1  0.433312      0      0   8.0500         2


## Simple Averaging Approach

In [47]:
# LogReg_clf = LogisticRegression()
# DTree_clf = DecisionTreeClassifier()
# SVC_clf = SVC()
# LogReg_clf.fit(X_train, y_train)
# DTree_clf.fit(X_train, y_train)
# SVC_clf.fit(X_train, y_train)
# LogReg_pred = LogReg_clf.predict(X_val)
# DTree_pred = DTree_clf.predict(X_val)
# SVC_pred = SVC_clf.predict(X_val)
# averaged_preds = (LogReg_pred + DTree_pred + SVC_pred)//3
# acc = accuracy_score(y_val, averaged_preds)
# print(acc)

# ==========================================
# 1. 建立三種不同的分類模型
# ==========================================

# Logistic Regression 增加迭代次數，改善收斂問題
LogReg_clf = LogisticRegression(max_iter=1000)

# Decision Tree：決策樹模型
DTree_clf = DecisionTreeClassifier(random_state=12)

# SVC：支援向量機分類器
SVC_clf = SVC()


# ==========================================
# 2. 使用訓練資料訓練三個模型
# ==========================================

# 訓練 Logistic Regression
LogReg_clf.fit(X_train, y_train)

# 訓練 Decision Tree
DTree_clf.fit(X_train, y_train)

# 訓練 Support Vector Machine
SVC_clf.fit(X_train, y_train)


# ==========================================
# 3. 使用驗證資料進行預測
# ==========================================

# Logistic Regression 的預測結果
LogReg_pred = LogReg_clf.predict(X_val)

# Decision Tree 的預測結果
DTree_pred = DTree_clf.predict(X_val)

# SVC 的預測結果
SVC_pred = SVC_clf.predict(X_val)


# ==========================================
# 4. 組合三個模型的預測結果
# ==========================================

# 將三個模型的預測結果相加，再除以 3（整數除法）
averaged_preds = (LogReg_pred + DTree_pred + SVC_pred) // 3


# ==========================================
# 5. 計算集成模型的準確率
# ==========================================

# 比較集成模型的預測與正確答案
acc = accuracy_score(y_val, averaged_preds)

# 印出 Accuracy
print(acc)


0.7222222222222222


## Bagging Classification Example

In [48]:
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    ExtraTreesClassifier
)
from sklearn.model_selection import KFold, cross_val_score

In [49]:
# def bagging_ensemble(model):
#     k_folds = KFold(n_splits=20, random_state=12,shuffle=True)
#     results = cross_val_score(model, X_train, y_train, cv=k_folds)
#     print(results.mean())


# ==========================================
# Bagging Classification
# 比較四種集成模型的分類準確率
# ==========================================

from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import KFold, cross_val_score

# 1. 建立 Logistic Regression Bagging 模型
# 使用 50 個 Logistic Regression 分類器
# 新版 sklearn 使用 estimator，而不是 base_estimator
logreg_bagging_model = BaggingClassifier(
    estimator=LogReg_clf,
    n_estimators=50,
    random_state=12
)

# 2. 建立 Decision Tree Bagging 模型
# 使用 50 個決策樹分類器
dtree_bagging_model = BaggingClassifier(
    estimator=DTree_clf,
    n_estimators=50,
    random_state=12
)

# 3. 建立 Random Forest 模型
# 使用 100 棵決策樹進行集成學習
random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=12
)

# 4. 建立 Extra Trees 模型
# 使用 100 棵高度隨機化的決策樹
extra_trees = ExtraTreesClassifier(
    n_estimators=100,
    random_state=12
)


# ==========================================
# 5. 建立 K-Fold 交叉驗證函數
# ==========================================

def bagging_ensemble(name, model):

    # 將資料分成 20 份
    # shuffle=True：先隨機打亂資料
    k_folds = KFold(
        n_splits=20,
        random_state=12,
        shuffle=True
    )

    # 執行 20-Fold Cross Validation
    # 每一輪使用 19 份訓練，1 份驗證
    results = cross_val_score(
        model,
        X_train,
        y_train,
        cv=k_folds,
        scoring='accuracy'
    )

    # 顯示模型名稱與平均準確率
    print(f"{name}: {results.mean():.4f}")


# ==========================================
# 6. 比較四種模型的 Accuracy
# ==========================================

bagging_ensemble("Logistic Regression Bagging", logreg_bagging_model)

bagging_ensemble("Decision Tree Bagging", dtree_bagging_model)

bagging_ensemble("Random Forest", random_forest)

bagging_ensemble("Extra Trees", extra_trees)


Logistic Regression Bagging: 0.7915
Decision Tree Bagging: 0.8189
Random Forest: 0.8101
Extra Trees: 0.7964


## Boosting Classification Example

In [50]:
# k_folds = KFold(n_splits=20, random_state=12,shuffle=True)
# num_estimators = [20, 40, 60, 80, 100]

# for i in num_estimators:
#     ada_boost = ???
#     results = cross_val_score(???,???,???,???)
#     print("Results for {} estimators:".format(i))
#     print(results.mean())

# ==========================================
# Boosting Classification Example
# 使用 AdaBoost 比較不同分類器數量的準確率
# ==========================================

# 1. 建立 20-Fold Cross Validation
# n_splits=20：將訓練資料分成 20 份
# random_state=12：固定隨機種子
# shuffle=True：先打亂資料再分割
k_folds = KFold(
    n_splits=20,
    random_state=12,
    shuffle=True
)

# 2. 設定要比較的分類器數量
# 分別測試 20、40、60、80、100 個弱分類器
num_estimators = [20, 40, 60, 80, 100]

# 3. 使用 for 迴圈測試不同的分類器數量
for i in num_estimators:

    # 建立 AdaBoost 分類模型
    # n_estimators=i：設定弱分類器的最大數量
    ada_boost = AdaBoostClassifier(
        n_estimators=i,
        random_state=12
    )

    # 使用 20-Fold Cross Validation 評估模型
    # 預設評分指標是 Accuracy
    results = cross_val_score(
        ada_boost,
        X_train,
        y_train,
        cv=k_folds
    )

    # 印出目前測試的分類器數量
    print("Results for {} estimators:".format(i))

    # 印出 20 次交叉驗證的平均 Accuracy
    print(results.mean())


Results for 20 estimators:
0.8064634146341463
Results for 40 estimators:
0.8089024390243903
Results for 60 estimators:
0.8051829268292684
Results for 80 estimators:
0.8039329268292683
Results for 100 estimators:
0.8051829268292684


## voting\Stacking Classification Example

In [52]:
# voting_clf = VotingClassifier(estimators=[('SVC', SVC_clf), ('DTree', DTree_clf), ('LogReg', LogReg_clf)], voting='hard')
# voting_clf.fit(X_train, y_train)
# preds = voting_clf.predict(X_val)
# acc = accuracy_score(y_val, preds)
# l_loss = log_loss(y_val, preds)
# f1 = f1_score(y_val, preds)

# print("Accuracy is: " + str(acc))
# print("Log Loss is: " + str(l_loss))
# print("F1 Score is: " + str(f1))


# ==========================================
# Voting Classification Example
# 使用三種不同模型進行 Hard Voting
# ==========================================

# 1. 建立 Voting 分類模型
# SVC：支援向量機
# DTree：決策樹
# LogReg：邏輯迴歸
# voting='hard'：採用多數決方式決定最終分類

voting_clf = VotingClassifier(
    estimators=[
        ('SVC', SVC_clf),
        ('DTree', DTree_clf),
        ('LogReg', LogReg_clf)
    ],
    voting='hard'
)

# 2. 使用訓練資料訓練三個分類器
voting_clf.fit(X_train, y_train)

# 3. 使用驗證資料進行預測
preds = voting_clf.predict(X_val)

# 4. 計算 Accuracy（整體預測正確率）
acc = accuracy_score(y_val, preds)

# 5. 計算 Log Loss
# 注意：這裡沿用老師以分類標籤計算的方式
l_loss = log_loss(y_val, preds)

# 6. 計算 F1-score（Precision 與 Recall 的調和平均）
f1 = f1_score(y_val, preds)

# 7. 顯示三種評估指標
print("Accuracy is: " + str(acc))
print("Log Loss is: " + str(l_loss))
print("F1 Score is: " + str(f1))


Accuracy is: 0.7888888888888889
Log Loss is: 7.609215715480287
F1 Score is: 0.6885245901639344


In [53]:
X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
715,3,1,-0.796286,0,0,7.6500,2
319,1,0,0.817561,1,1,134.5000,0
829,1,0,2.508257,0,0,80.0000,2
79,3,0,0.049062,0,0,12.4750,2
484,1,1,-0.335187,1,0,91.0792,0
...,...,...,...,...,...,...,...
241,3,0,-0.104637,1,0,15.5000,1
253,3,1,0.049062,1,0,16.1000,2
390,1,1,0.510161,1,2,120.0000,2
667,3,1,-0.104637,0,0,7.7750,2


In [54]:
testing_data

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,892,3,1,34.5,0,0,7.8292,1
1,893,3,0,47.0,1,0,7.0000,2
2,894,2,1,62.0,0,0,9.6875,1
3,895,3,1,27.0,0,0,8.6625,2
4,896,3,0,22.0,1,1,12.2875,2
...,...,...,...,...,...,...,...,...
413,1305,3,1,27.0,0,0,8.0500,2
414,1306,1,0,39.0,0,0,108.9000,0
415,1307,3,1,38.5,0,0,7.2500,2
416,1308,3,1,27.0,0,0,8.0500,2


In [55]:

# 移除測試資料中的 PassengerId（乘客編號）欄位
# axis=1 表示刪除欄位（Column）
# 將處理後的資料儲存到 test 變數
test = testing_data.drop(labels=['PassengerId'], axis=1)

# 查看測試資料的維度（資料筆數、特徵數量）
test.shape


(418, 7)

In [58]:

# ==========================================
# Ensemble Learning - Voting Classifier
# 將 Titanic 生還預測結果輸出為 CSV 檔案
# ==========================================

# 1. 建立新的 DataFrame，設定兩個欄位
# PassengerId：乘客編號
# Survived：生還結果（0 = 未生還，1 = 生還）
submission_df = pd.DataFrame(columns=['PassengerId', 'Survived'])

# 2. 從測試資料取得 PassengerId
# 將乘客編號複製到 submission_df
submission_df['PassengerId'] = testing_data['PassengerId']

# 3. 使用訓練完成的 Voting Classifier 進行預測
# test：已完成資料前處理的測試資料
# predict()：預測每位乘客是否生還
preds = voting_clf.predict(test)

# 4. 將 AI 模型的預測結果填入 Survived 欄位
submission_df['Survived'] = preds

# 5. 將 DataFrame 輸出成 CSV 檔案
# submissions.csv：輸出的檔案名稱
# header=True：保留 PassengerId、Survived 欄位名稱
# index=False：不輸出 DataFrame 的索引
submission_df.to_csv('submissions.csv', header=True, index=False)


